# WireMod detector unisim (Products A + B)

Match at **`sel_all`**, then for **each universe** (calo ± and efield) walk the
full selection pipeline on matched files:

1. **Product A** — selection / cut-stage variables  
2. **Product B** — measurement variables at final (`2prong-mup`) stage  

Per geometry (YZ, XTXW), the **final** WireMod unisim is the **total envelope**:
max |shift| over all `ccal/alpha/beta/R` ± universes **and** `efield` vs CV.

**Requires** matched files with `evt_cv` + eight calo tables + `evt_efield`
(`sel_all-updatecalo.py`).

**Final outputs** (loaded by `systematics-detector.ipynb` / summary):
- `WireMod/DetectorSelection/detector_sel_syst_dict.npz` (A)
- `WireMod/Detector/detector_syst_dict.npz` (B)

**Inspection only** (not used downstream): per-component envelopes under
`WireMod/inspect_envelopes/`.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path("/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana")
sys.path.insert(0, str(REPO))

from analysis_village.numucc_1p0pi.dataset_locations import PLOTS_BASE, SPRING_GEN1_ROOT
from analysis_village.numucc_1p0pi.scripts import dent_compare as dc
from analysis_village.numucc_1p0pi.syst_disk_layout import (
    FILE_DETECTOR,
    FILE_DETECTOR_SEL,
    SUB_DETECTOR,
    SUB_DETECTOR_SEL,
)
from analysis_village.numucc_1p0pi.syst_detvar_common import (
    WIREMOD_ENVELOPE_SHIFTED,
    accumulate_wiremod_matched_products,
    assert_variations_matched,
    assert_wiremod_universes,
    build_wiremod_detector_dict,
    frac_unc_pct_from_pack,
    glob_matched_dfs,
    load_or_build_cache,
    log,
    pot_scales_to_cv,
    save_detector_npz,
    wiremod_component_shifted_univs,
    wiremod_geometry_hists_for_envelope,
)
from analysis_village.numucc_1p0pi.utils import dpi
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig


In [ ]:
DFS = Path(os.environ.get("NUMUCC_SPRING_GEN1_ROOT", SPRING_GEN1_ROOT))

WIREMOD_DIRS = {
    "YZ": DFS / os.environ.get("WIREMOD_YZ_SEL_ALL", "SET_ME__sel_all-mc-WireModYZ"),
    "XTXW": DFS / os.environ.get("WIREMOD_XTXW_SEL_ALL", "SET_ME__sel_all-mc-WireModXTXW"),
    "CV": DFS / os.environ.get("WIREMOD_CV_SEL_ALL", "SET_ME__sel_all-mc-calovar"),
}

OUT_BASE = Path(os.environ.get("WIREMOD_OUT_BASE", str(PLOTS_BASE / "systematics-final" / "WireMod")))
CACHE_DIR = OUT_BASE / "cache"
FIG_DIR = OUT_BASE / "plots"
DET_B_DIR = OUT_BASE / SUB_DETECTOR
DET_A_DIR = OUT_BASE / SUB_DETECTOR_SEL
# Component envelopes only — never under Detector/ (downstream loaders ignore this).
INSPECT_DIR = OUT_BASE / "inspect_envelopes"
INSPECT_NPZ_DIR = INSPECT_DIR / "npz"
INSPECT_FIG_DIR = INSPECT_DIR / "plots"

CACHE_PATH = CACHE_DIR / "wiremod_sel_all_products.pkl"
NPZ_B = DET_B_DIR / FILE_DETECTOR
NPZ_A = DET_A_DIR / FILE_DETECTOR_SEL

FORCE_REPROCESS = os.environ.get("WIREMOD_FORCE_REPROCESS", "0") == "1"
ASSERT_MAX_FILES = int(os.environ.get("WIREMOD_ASSERT_MAX_FILES", "20"))
SAVE_FIGS = True

for d in (CACHE_DIR, FIG_DIR, DET_A_DIR, DET_B_DIR, INSPECT_NPZ_DIR, INSPECT_FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

VARIATIONS = {
    lab: glob_matched_dfs(p, filename_str="sel_all")
    for lab, p in WIREMOD_DIRS.items()
    if Path(p).is_dir()
}
for lab, files in VARIATIONS.items():
    print(f"{lab}: {len(files)} matched sel_all files")
print("cache:", CACHE_PATH)
print("Product A (final):", NPZ_A)
print("Product B (final):", NPZ_B)
print("inspect only:", INSPECT_DIR)
print("total envelope univs:", WIREMOD_ENVELOPE_SHIFTED)


In [ ]:
assert_variations_matched(
    {k: v for k, v in VARIATIONS.items() if v},
    max_files_per_var=ASSERT_MAX_FILES,
    format="sel_all",
)
# Calo ± + efield required on every geometry sample
for lab, files in VARIATIONS.items():
    if not files:
        continue
    log(f"checking WireMod universes (calo + efield): {lab}")
    assert_wiremod_universes(files, require_efield=True)


In [ ]:
FINAL_VAR_DEFS = dc.build_final_var_defs()
print(f"final vars: {len(FINAL_VAR_DEFS)}")


def _build_wiremod_cache():
    by_geom = {}
    pot_by = {}
    for lab, files in VARIATIONS.items():
        if not files:
            continue
        log(f"[{lab}] walk {len(files)} matched sel_all+calo+efield files …")
        prod = accumulate_wiremod_matched_products(
            files,
            final_var_defs=FINAL_VAR_DEFS,
            include_cut_stage=True,
        )
        by_geom[lab] = prod
        pot_by[lab] = prod["pot"]
        log(f"  {lab} POT={prod['pot']:.3e} univs={prod['universes']}")
    pot_scales = pot_scales_to_cv(pot_by, reference="CV") if "CV" in pot_by else {k: 1.0 for k in pot_by}
    # Scale all universe hists onto CV POT
    for lab, sc in pot_scales.items():
        if lab not in by_geom or abs(float(sc) - 1.0) < 1e-15:
            continue
        sc = float(sc)
        for univ, payload in by_geom[lab]["by_universe"].items():
            for key in ("hists_cut", "hists_final"):
                payload[key] = {k: np.asarray(v, dtype=float) * sc for k, v in payload[key].items()}
    return {
        "by_geom": by_geom,
        "pot_by_variation": pot_by,
        "pot_scales": pot_scales,
        "match_stage": "sel_all",
        "envelope": "calo_plus_efield",
        "variations": {k: list(v) for k, v in VARIATIONS.items()},
    }


payload = load_or_build_cache(CACHE_PATH, _build_wiremod_cache, force=FORCE_REPROCESS)
by_geom = payload["by_geom"]
print("POT scales:", payload.get("pot_scales"))
for lab, prod in by_geom.items():
    print(lab, "cut vars", len(prod["cut_var_names"]), "final vars", len(prod["final_var_names"]), "univs", prod.get("universes"))


## Final WireMod products (calo + efield total envelope)

These NPZs under `Detector/` / `DetectorSelection/` are what
`systematics-detector.ipynb` and `systematics-summary.ipynb` load.


In [ ]:
def _all_hists_for_product(product: str):
    # {geometry: {univ: {var: hist}}}
    out = {}
    for lab, prod in by_geom.items():
        if lab == "CV":
            # CV sample is POT reference only; envelope uses YZ / XTXW
            continue
        out[lab] = wiremod_geometry_hists_for_envelope(prod["by_universe"], product=product)
    return out


all_hists_a = _all_hists_for_product("cut")
all_hists_b = _all_hists_for_product("final")

cut_names = next(iter(by_geom.values()))["cut_var_names"]
final_names = next(iter(by_geom.values()))["final_var_names"]

dict_a = build_wiremod_detector_dict(
    all_hists_a, cut_names, wiremod_labels=("YZ", "XTXW"), shifted_univs=WIREMOD_ENVELOPE_SHIFTED
)
dict_b = build_wiremod_detector_dict(
    all_hists_b, final_names, wiremod_labels=("YZ", "XTXW"), shifted_univs=WIREMOD_ENVELOPE_SHIFTED
)

save_detector_npz(
    dict_a,
    NPZ_A,
    manifest={
        "source": "WireMod",
        "product": "A_selection",
        "match_stage": "sel_all",
        "method": "total_envelope_calo_plus_efield",
        "shifted_univs": list(WIREMOD_ENVELOPE_SHIFTED),
        "cache": str(CACHE_PATH),
        "n_vars": len(dict_a.get("detector", {})),
    },
)
save_detector_npz(
    dict_b,
    NPZ_B,
    manifest={
        "source": "WireMod",
        "product": "B_measurement",
        "match_stage": "sel_all",
        "method": "total_envelope_calo_plus_efield",
        "shifted_univs": list(WIREMOD_ENVELOPE_SHIFTED),
        "cache": str(CACHE_PATH),
        "n_vars": len(dict_b.get("detector", {})),
    },
)
print("Product A vars:", len(dict_a.get("detector", {})))
print("Product B vars:", len(dict_b.get("detector", {})))


## Quick inspect — final total envelope (Product B)


In [ ]:
for vc in (VariableConfig.all_events(), VariableConfig.muon_momentum()):
    vsn = vc.var_save_name
    fig, ax = plt.subplots(figsize=(6.2, 4.2))
    for tag, color, label in (
        ("wiremod_yz", "C0", "WireMod YZ (calo+efield env)"),
        ("wiremod_xtxw", "C1", "WireMod XTXW (calo+efield env)"),
    ):
        pack = dict_b.get(f"detector-{tag}", {}).get(vsn)
        if pack is None:
            continue
        w = frac_unc_pct_from_pack(pack)
        ax.hist(vc.bin_centers, bins=vc.bins, weights=w, histtype="step", lw=1.8, color=color, label=label)
    ax.set_ylabel("Uncertainty [%]")
    ax.set_xlabel(vc.var_labels[0] if vc.var_labels else vsn)
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    if SAVE_FIGS:
        fig.savefig(FIG_DIR / f"wiremod_total_env_frac_unc__{vsn}.png", dpi=dpi, bbox_inches="tight")
    plt.show()


## Inspect — envelopes from each calo parameter and efield alone

Diagnostic only. Written under `inspect_envelopes/` so they are **not** confused
with the final `Detector/` / `DetectorSelection/` NPZs used by
`systematics-detector.ipynb` and `systematics-summary.ipynb`.

For each calo param: envelope of that param's `±` pair only.  
For efield: unisim of `efield` vs CV.


In [ ]:
COMPONENT_SHIFTED = wiremod_component_shifted_univs()
print("component envelopes:", {k: list(v) for k, v in COMPONENT_SHIFTED.items()})

inspect_dicts_b = {}
inspect_summary = {"product": "B_measurement", "note": "inspection only — not for summary/detector combine", "components": {}}

for comp, shifted in COMPONENT_SHIFTED.items():
    d = build_wiremod_detector_dict(
        all_hists_b,
        final_names,
        wiremod_labels=("YZ", "XTXW"),
        shifted_univs=shifted,
    )
    inspect_dicts_b[comp] = d
    out_npz = INSPECT_NPZ_DIR / f"wiremod_envelope_{comp}_productB.npz"
    save_detector_npz(
        d,
        out_npz,
        manifest={
            "source": "WireMod",
            "product": "B_measurement_inspect",
            "component": comp,
            "method": "component_envelope",
            "shifted_univs": list(shifted),
            "not_for_downstream": True,
            "cache": str(CACHE_PATH),
        },
    )
    # Integrated frac unc [%] per geometry + combined, when available
    row = {"shifted_univs": list(shifted)}
    for key in ("detector-wiremod_yz", "detector-wiremod_xtxw", "detector"):
        pack = d.get(key, {}).get("integrated")
        if pack is None:
            continue
        w = frac_unc_pct_from_pack(pack)
        row[key] = float(np.asarray(w).ravel()[0]) if len(np.asarray(w).ravel()) else float("nan")
    inspect_summary["components"][comp] = row
    print(f"  {comp}: wrote {out_npz.name}  integrated={row}")

summary_path = INSPECT_DIR / "component_envelope_summary.json"
summary_path.write_text(json.dumps(inspect_summary, indent=2))
print("wrote", summary_path)


In [ ]:
# Overlay component envelopes vs total (Product B) for a few vars
COMP_COLORS = {
    "ccal": "C2",
    "alpha": "C3",
    "beta": "C4",
    "R": "C5",
    "efield": "C6",
}

for vc in (VariableConfig.all_events(), VariableConfig.muon_momentum()):
    vsn = vc.var_save_name
    for tag, geom_label in (("wiremod_yz", "YZ"), ("wiremod_xtxw", "XTXW")):
        fig, ax = plt.subplots(figsize=(6.8, 4.4))
        # total envelope (final)
        pack_tot = dict_b.get(f"detector-{tag}", {}).get(vsn)
        if pack_tot is not None:
            ax.hist(
                vc.bin_centers,
                bins=vc.bins,
                weights=frac_unc_pct_from_pack(pack_tot),
                histtype="step",
                lw=2.2,
                color="k",
                label="total (calo+efield)",
            )
        for comp, d in inspect_dicts_b.items():
            pack = d.get(f"detector-{tag}", {}).get(vsn)
            if pack is None:
                continue
            ax.hist(
                vc.bin_centers,
                bins=vc.bins,
                weights=frac_unc_pct_from_pack(pack),
                histtype="step",
                lw=1.5,
                color=COMP_COLORS.get(comp, None),
                label=comp,
            )
        ax.set_title(f"WireMod {geom_label} — component vs total envelope")
        ax.set_ylabel("Uncertainty [%]")
        ax.set_xlabel(vc.var_labels[0] if vc.var_labels else vsn)
        ax.set_ylim(bottom=0)
        ax.legend(fontsize=7, ncol=2)
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        if SAVE_FIGS:
            fig.savefig(
                INSPECT_FIG_DIR / f"wiremod_components_vs_total__{geom_label}__{vsn}.png",
                dpi=dpi,
                bbox_inches="tight",
            )
        plt.show()


In [ ]:
# Also dump Product A component envelopes to the same inspect tree
inspect_summary_a = {"product": "A_selection", "note": "inspection only", "components": {}}
for comp, shifted in COMPONENT_SHIFTED.items():
    d = build_wiremod_detector_dict(
        all_hists_a,
        cut_names,
        wiremod_labels=("YZ", "XTXW"),
        shifted_univs=shifted,
    )
    out_npz = INSPECT_NPZ_DIR / f"wiremod_envelope_{comp}_productA.npz"
    save_detector_npz(
        d,
        out_npz,
        manifest={
            "source": "WireMod",
            "product": "A_selection_inspect",
            "component": comp,
            "method": "component_envelope",
            "shifted_univs": list(shifted),
            "not_for_downstream": True,
        },
    )
    row = {"shifted_univs": list(shifted)}
    for key in ("detector-wiremod_yz", "detector-wiremod_xtxw", "detector"):
        pack = d.get(key, {}).get("integrated")
        if pack is None:
            continue
        w = frac_unc_pct_from_pack(pack)
        row[key] = float(np.asarray(w).ravel()[0]) if len(np.asarray(w).ravel()) else float("nan")
    inspect_summary_a["components"][comp] = row

path_a = INSPECT_DIR / "component_envelope_summary_productA.json"
path_a.write_text(json.dumps(inspect_summary_a, indent=2))
print("wrote", path_a)
print("inspect tree:", INSPECT_DIR)
